In [0]:
%sql
-- Distinct values + counts for order_status
SELECT order_status, COUNT(*) 
FROM ecommerce.base.orders 
GROUP BY order_status 
ORDER BY COUNT(*) DESC;

order_status,COUNT(*)
Completed,106894
Returned,7000
Cancelled,4881
Pending,1225


In [0]:
%sql
-- Distinct payment_method values (dictionary flags inconsistent spelling)
SELECT payment_method, COUNT(*) 
FROM ecommerce.base.payments 
GROUP BY payment_method 
ORDER BY COUNT(*) DESC;

payment_method,COUNT(*)
Credit Card,47831
Debit Card,26116
Digital Wallet,18958
PayPal,16915
Bank Transfer,9580
Credit Card,86
CREDIT CARD,82
credit card,78
DEBIT CARD,53
Debit Card,42


In [0]:
%sql
-- Distinct city values in customers (dictionary flags inconsistency)
SELECT city, COUNT(*) 
FROM ecommerce.base.customers 
GROUP BY city 
ORDER BY COUNT(*) DESC;

city,COUNT(*)
null,45
North Michael,21
East Michael,18
South James,14
West Michael,14
East James,12
Michaelmouth,12
South Michael,12
New Jennifer,12
Johnsonland,11


In [0]:
%sql
SELECT
    LOWER(TRIM(city)) AS standardized_city,
    COUNT(DISTINCT city) AS variations,
    COUNT(*) AS total_rows
FROM ecommerce.base.customers
WHERE city IS NOT NULL
GROUP BY LOWER(TRIM(city))
HAVING COUNT(DISTINCT city) > 1
ORDER BY variations DESC;

standardized_city,variations,total_rows
julieview,2,2
brianmouth,2,2
east randy,2,2
kevintown,2,2
north tammy,2,2
kyleport,2,2
nicholasbury,2,2
east cody,2,3
smithton,2,2
new kaitlyn,2,2


In [0]:
%sql
-- Invalid birth_date check (future dates, absurdly old dates)
SELECT customer_id, birth_date
FROM ecommerce.base.customers
WHERE birth_date > CURRENT_DATE
   OR birth_date < DATE '1900-01-01';

customer_id,birth_date
2975,2999-01-01
5702,2999-01-01
5716,2999-01-01
6305,2999-01-01
6683,2999-01-01
7554,2999-01-01
8333,2999-01-01
9939,2999-01-01
10571,2999-01-01
10842,2999-01-01


In [0]:
%sql
-- Returns table profiling: nulls, return_reason distribution, refund_amount range
SELECT
  COUNT(*) AS total_returns,
  COUNT(*) - COUNT(return_reason) AS null_reason,
  COUNT(*) - COUNT(refund_amount) AS null_refund,
  MIN(refund_amount) AS min_refund,
  MAX(refund_amount) AS max_refund,
  AVG(refund_amount) AS avg_refund
FROM ecommerce.base.returns;

total_returns,null_reason,null_refund,min_refund,max_refund,avg_refund
15050,0,0,5.37,2114.22,234.90098737541592


In [0]:
%sql
-- Distinct return_reason values and their frequency
SELECT return_reason, COUNT(*) AS occurrences
FROM ecommerce.base.returns
GROUP BY return_reason
ORDER BY occurrences DESC;

return_reason,occurrences
Size Issue,3643
Damaged,2525
Customer Changed Mind,2431
Poor Quality,2187
Wrong Item,2098
Late Delivery,2016
size issue,38
customer changed mind,27
poor quality,25
wrong item,24


In [0]:
%sql
-- Payment status distribution + mismatch check against orders.order_total
SELECT
  p.payment_status,
  COUNT(*) AS occurrences,
  SUM(CASE WHEN p.amount != o.order_total THEN 1 ELSE 0 END) AS amount_mismatch_count
FROM ecommerce.base.payments p
JOIN ecommerce.base.orders o ON p.order_id = o.order_id
GROUP BY p.payment_status
ORDER BY occurrences DESC;

payment_status,occurrences,amount_mismatch_count
Successful,113933,0
Failed,4094,0
Refunded,1973,0
